In [ ]:
# Configuration du chemin du projet pour permettre les imports depuis src/
from pathlib import Path
import sys

project_root = None
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "src").is_dir():
        project_root = candidate
        break

if project_root is None:
    raise FileNotFoundError(
        "Dossier src introuvable. Ouvre le notebook depuis le dossier du projet "
        "options-pricer-delta-hedging."
    )

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.pricing import (
    black_scholes_call,
    monte_carlo_call,
    monte_carlo_call_antithetic
)

S0 = 100
K = 100
T = 1
r = 0.05
sigma = 0.20

prix_reference = black_scholes_call(S0, K, T, r, sigma)

print("Prix Black-Scholes :", prix_reference)



In [ ]:
n_values = [100, 500, 1000, 5000, 10000, 50000, 100000]

results = []

for N in n_values:

    # Monte-Carlo standard
    standard = monte_carlo_call(
        S0, K, T, r, sigma,
        n_simulations=N,
        seed=42
    )

    # Monte-Carlo antithétique
    antithetic = monte_carlo_call_antithetic(
        S0, K, T, r, sigma,
        n_pairs=N // 2,
        seed=42
    )

    results.append({
        "N": N,
        "Prix standard": standard["price"],
        "Prix antithétique": antithetic["price"],
        "Erreur standard": standard["standard_error"],
        "Erreur antithétique": antithetic["standard_error"]
    })

df = pd.DataFrame(results)

display(df)

In [ ]:

plt.figure(figsize=(11, 6))

plt.plot(
    df["N"],
    df["Prix standard"],
    marker="o",
    label="Monte-Carlo standard"
)

plt.plot(
    df["N"],
    df["Prix antithétique"],
    marker="s",
    label="Variables antithétiques"
)

plt.axhline(
    y=prix_reference,
    linestyle="--",
    label="Black-Scholes"
)

plt.xscale("log")

plt.xlabel("Nombre total de trajectoires")
plt.ylabel("Prix estimé du Call (€)")

plt.title("Convergence des estimateurs Monte-Carlo")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("../figures/convergence_monte_carlo.png", dpi=300)
plt.show()


In [ ]:
from src.pricing import monte_carlo_asian_call

asian = monte_carlo_asian_call(
    S0=100,
    K=100,
    T=1,
    r=0.05,
    sigma=0.20,
    n_simulations=50_000,
    n_steps=252,
    seed=42
)

print("Prix du Call asiatique :", asian["price"])
print("Erreur standard :", asian["standard_error"])
print("Intervalle de confiance :", asian["confidence_interval"])

In [ ]:
from src.pricing import monte_carlo_asian_call

asian = monte_carlo_asian_call(
    S0=100,
    K=100,
    T=1,
    r=0.05,
    sigma=0.20,
    n_simulations=50_000,
    n_steps=252,
    seed=42
)

print("PRICING OPTION ASIATIQUE")
print("------------------------")
print("Prix :", round(asian["price"], 6))
print("Erreur standard :", round(asian["standard_error"], 6))
print("IC 95 % :", asian["confidence_interval"])

In [ ]:
european = black_scholes_call(
    100, 100, 1, 0.05, 0.20
)

print("Call européen :", round(european, 6))
print("Call asiatique :", round(asian["price"], 6))

print(
    "Différence :",
    round(european - asian["price"], 6)
)

In [ ]:
from src.pricing import (
    black_scholes_call,
    monte_carlo_asian_call
)

# Paramètres du marché
S0 = 100
K = 100
T = 1
r = 0.05
sigma = 0.20

# Prix européen
european = black_scholes_call(
    S0, K, T, r, sigma
)

# Prix asiatique
asian = monte_carlo_asian_call(
    S0, K, T, r, sigma,
    n_simulations=100_000,
    n_steps=252,
    seed=42
)

print("PRICING COMPARATIF")
print("--------------------------")
print("Call européen :", round(european, 6))
print("Call asiatique :", round(asian["price"], 6))
print("Erreur standard :", round(asian["standard_error"], 6))
print("IC 95 % :", asian["confidence_interval"])
print("Différence :", round(european - asian["price"], 6))

In [ ]:
steps_values = [12, 52, 252]

asian_results = []

for steps in steps_values:

    result = monte_carlo_asian_call(
        S0, K, T, r, sigma,
        n_simulations=100_000,
        n_steps=steps,
        seed=42
    )

    asian_results.append({
        "Observations": steps,
        "Prix asiatique": result["price"],
        "Erreur standard": result["standard_error"]
    })

df_asian = pd.DataFrame(asian_results)

display(df_asian)

In [ ]:
from src.greeks import delta_call, delta_put

S0 = 100
K = 100
T = 1
r = 0.05
sigma = 0.20

call_delta = delta_call(S0, K, T, r, sigma)
put_delta = delta_put(S0, K, T, r, sigma)

print("CALL DELTA :", call_delta)
print("PUT DELTA :", put_delta)
print("Somme des Deltas :", call_delta - put_delta)

In [ ]:
from src.greeks import gamma

gamma_value = gamma(100, 100, 1, 0.05, 0.20)

print("GAMMA :", gamma_value)

In [ ]:
from src.pricing import black_scholes_call

S0 = 100
K = 100
T = 1
r = 0.05
sigma = 0.20

h = 0.01

gamma_numerical = (
    black_scholes_call(S0+h, K, T, r, sigma)
    - 2*black_scholes_call(S0, K, T, r, sigma)
    + black_scholes_call(S0-h, K, T, r, sigma)
) / h**2

gamma_analytical = gamma(S0, K, T, r, sigma)

print("Gamma analytique :", gamma_analytical)
print("Gamma numérique :", gamma_numerical)
print("Erreur :", abs(gamma_analytical-gamma_numerical))

In [ ]:
from src.greeks import vega

vega_value = vega(100, 100, 1, 0.05, 0.20)

print("VEGA :", vega_value)
print("VEGA par point de volatilité :", vega_value / 100)

In [ ]:
from src.pricing import black_scholes_call
from src.greeks import vega

S0 = 100
K = 100
T = 1
r = 0.05
sigma = 0.20

h = 0.0001

vega_numerical = (
    black_scholes_call(S0, K, T, r, sigma+h)
    - black_scholes_call(S0, K, T, r, sigma-h)
) / (2*h)

vega_analytical = vega(S0, K, T, r, sigma)

print("Vega analytique :", vega_analytical)
print("Vega numérique :", vega_numerical)
print("Erreur :", abs(vega_analytical-vega_numerical))

In [ ]:
tc = greeks.theta_call(100, 100, 1, 0.05, 0.20)
tp = greeks.theta_put(100, 100, 1, 0.05, 0.20)

print("THETA CALL :", tc)
print("THETA PUT :", tp)

In [ ]:

from src.pricing import black_scholes_call, black_scholes_put
from src.greeks import theta_call, theta_put

S0 = 100
K = 100
T = 1
r = 0.05
sigma = 0.20

h = 0.0001

# Theta numérique du Call
theta_call_num = (
    black_scholes_call(S0, K, T-h, r, sigma)
    - black_scholes_call(S0, K, T+h, r, sigma)
) / (2*h)

# Theta numérique du Put
theta_put_num = (
    black_scholes_put(S0, K, T-h, r, sigma)
    - black_scholes_put(S0, K, T+h, r, sigma)
) / (2*h)

# Theta analytique
theta_call_ana = theta_call(S0, K, T, r, sigma)
theta_put_ana = theta_put(S0, K, T, r, sigma)

print("CALL")
print("Analytique :", theta_call_ana)
print("Numérique :", theta_call_num)
print("Erreur :", abs(theta_call_ana-theta_call_num))

print("\nPUT")
print("Analytique :", theta_put_ana)
print("Numérique :", theta_put_num)
print("Erreur :", abs(theta_put_ana-theta_put_num))


In [ ]:
from src.greeks import rho_call, rho_put

rho_c = rho_call(100, 100, 1, 0.05, 0.20)
rho_p = rho_put(100, 100, 1, 0.05, 0.20)

print("RHO CALL :", rho_c)
print("RHO PUT :", rho_p)

print("RHO CALL par 1% :", rho_c / 100)
print("RHO PUT par 1% :", rho_p / 100)

In [ ]:

from src.pricing import black_scholes_call, black_scholes_put
from src.greeks import rho_call, rho_put

S0 = 100
K = 100
T = 1
r = 0.05
sigma = 0.20

h = 0.0001

# Rho numérique Call
rho_call_num = (
    black_scholes_call(S0, K, T, r+h, sigma)
    - black_scholes_call(S0, K, T, r-h, sigma)
) / (2*h)

# Rho numérique Put
rho_put_num = (
    black_scholes_put(S0, K, T, r+h, sigma)
    - black_scholes_put(S0, K, T, r-h, sigma)
) / (2*h)

# Rho analytique
rho_call_ana = rho_call(S0, K, T, r, sigma)
rho_put_ana = rho_put(S0, K, T, r, sigma)

print("CALL")
print("Analytique :", rho_call_ana)
print("Numérique :", rho_call_num)
print("Erreur :", abs(rho_call_ana-rho_call_num))

print("\nPUT")
print("Analytique :", rho_put_ana)
print("Numérique :", rho_put_num)
print("Erreur :", abs(rho_put_ana-rho_put_num))


In [ ]:
from src.greeks import calculate_greeks
import pandas as pd

call_greeks = calculate_greeks(
    100, 100, 1, 0.05, 0.20,
    option_type="call"
)

put_greeks = calculate_greeks(
    100, 100, 1, 0.05, 0.20,
    option_type="put"
)

df_greeks = pd.DataFrame({
    "Call": call_greeks,
    "Put": put_greeks
})

display(df_greeks)

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

from src.greeks import delta_call, delta_put

# Paramètres
K = 100
T = 1
r = 0.05
sigma = 0.20

# Prix du sous-jacent
S_values = np.linspace(60, 140, 200)

# Calcul des Deltas
call_deltas = [
    delta_call(S, K, T, r, sigma)
    for S in S_values
]

put_deltas = [
    delta_put(S, K, T, r, sigma)
    for S in S_values
]

# Graphique
plt.figure(figsize=(10, 6))

plt.plot(
    S_values,
    call_deltas,
    label="Delta Call"
)

plt.plot(
    S_values,
    put_deltas,
    label="Delta Put"
)

plt.axvline(
    K,
    linestyle="--",
    label="Strike K"
)

plt.axhline(0, linestyle=":", alpha=0.5)

plt.xlabel("Prix du sous-jacent S")
plt.ylabel("Delta")

plt.title("Profils du Delta - Call et Put")

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig("../figures/delta_profile.png", dpi=300)
plt.show()


In [ ]:

import numpy as np
import matplotlib.pyplot as plt

from src.greeks import gamma

# Paramètres
K = 100
T = 1
r = 0.05
sigma = 0.20

# Prix du sous-jacent
S_values = np.linspace(60, 140, 200)

# Calcul du Gamma
gamma_values = [
    gamma(S, K, T, r, sigma)
    for S in S_values
]

# Graphique
plt.figure(figsize=(10, 6))

plt.plot(
    S_values,
    gamma_values,
    label="Gamma"
)

plt.axvline(
    K,
    linestyle="--",
    label="Strike K"
)

plt.xlabel("Prix du sous-jacent S")
plt.ylabel("Gamma")

plt.title("Profil du Gamma - Option européenne")

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig("../figures/gamma_profile.png", dpi=300)
plt.show()


In [ ]:

import numpy as np
import matplotlib.pyplot as plt

from src.greeks import vega

# Paramètres
K = 100
T = 1
r = 0.05
sigma = 0.20

# Prix du sous-jacent
S_values = np.linspace(60, 140, 200)

# Calcul du Vega
vega_values = [
    vega(S, K, T, r, sigma)
    for S in S_values
]

# Graphique
plt.figure(figsize=(10, 6))

plt.plot(
    S_values,
    vega_values,
    label="Vega"
)

plt.axvline(
    K,
    linestyle="--",
    label="Strike K"
)

plt.xlabel("Prix du sous-jacent S")
plt.ylabel("Vega")

plt.title("Profil du Vega - Option européenne")

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig("../figures/vega_profile.png", dpi=300)
plt.show()


In [ ]:

import numpy as np
import matplotlib.pyplot as plt

from src.greeks import theta_call, theta_put

# Paramètres
K = 100
T = 1
r = 0.05
sigma = 0.20

# Prix du sous-jacent
S_values = np.linspace(60, 140, 200)

# Calcul des Theta
call_theta_values = [
    theta_call(S, K, T, r, sigma)
    for S in S_values
]

put_theta_values = [
    theta_put(S, K, T, r, sigma)
    for S in S_values
]

# Graphique
plt.figure(figsize=(10, 6))

plt.plot(
    S_values,
    call_theta_values,
    label="Theta Call"
)

plt.plot(
    S_values,
    put_theta_values,
    label="Theta Put"
)

plt.axvline(
    K,
    linestyle="--",
    label="Strike K"
)

plt.axhline(0, linestyle=":", alpha=0.5)

plt.xlabel("Prix du sous-jacent S")
plt.ylabel("Theta annuel")

plt.title("Profils du Theta - Call et Put")

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig("../figures/theta_profile.png", dpi=300)
plt.show()


In [ ]:

import numpy as np
import matplotlib.pyplot as plt

from src.greeks import theta_call, theta_put

# Paramètres
S0 = 100
K = 100
r = 0.05
sigma = 0.20

# Maturités en années
T_values = np.linspace(0.05, 2, 200)

# Calcul des Theta
call_theta_values = [
    theta_call(S0, K, T, r, sigma)
    for T in T_values
]

put_theta_values = [
    theta_put(S0, K, T, r, sigma)
    for T in T_values
]

# Graphique
plt.figure(figsize=(10, 6))

plt.plot(
    T_values,
    call_theta_values,
    label="Theta Call"
)

plt.plot(
    T_values,
    put_theta_values,
    label="Theta Put"
)

plt.axhline(0, linestyle=":", alpha=0.5)

plt.xlabel("Maturité T (années)")
plt.ylabel("Theta annuel")

plt.title("Évolution du Theta selon la maturité")

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig("../figures/theta_maturity.png", dpi=300)
plt.show()


In [ ]:
from src.pricing import black_scholes_call

# Paramètres du marché
S0 = 100
K = 100
T = 1
r = 0.05

# Volatilité réelle utilisée pour générer le prix
true_sigma = 0.25

# Prix de marché simulé
market_price = black_scholes_call(
    S0, K, T, r, true_sigma
)

print("Prix du Call observé :", market_price)

In [ ]:
implied_sigma = implied_volatility_call(
    market_price=market_price,
    S0=S0,
    K=K,
    T=T,
    r=r,
    initial_sigma=0.20
)

print("Volatilité réelle :", true_sigma)
print("Volatilité implicite :", implied_sigma)
print("Erreur :", abs(true_sigma - implied_sigma))

In [ ]:
import importlib
import src.volatility as volatility

importlib.reload(volatility)

In [ ]:
implied_sigma = volatility.implied_volatility_call(
    market_price=market_price,
    S0=S0,
    K=K,
    T=T,
    r=r,
    initial_sigma=0.20
)

print("Volatilité réelle :", true_sigma)
print("Volatilité implicite :", implied_sigma)
print("Erreur :", abs(true_sigma - implied_sigma))

In [ ]:
from src.pricing import black_scholes_put

S0 = 100
K = 100
T = 1
r = 0.05

true_sigma_put = 0.30

market_put_price = black_scholes_put(
    S0, K, T, r, true_sigma_put
)

print("Prix du Put observé :", market_put_price)

In [ ]:
from src.volatility import implied_volatility_put

implied_sigma_put = implied_volatility_put(
    market_price=market_put_price,
    S0=S0,
    K=K,
    T=T,
    r=r,
    initial_sigma=0.20
)

print("\nVolatilité réelle :", true_sigma_put)
print("Volatilité implicite :", implied_sigma_put)

print(
    "Erreur :",
    abs(true_sigma_put - implied_sigma_put)
)

In [ ]:
import importlib
import src.volatility as volatility

importlib.reload(volatility)

implied_sigma_put = volatility.implied_volatility_put(
    market_price=market_put_price,
    S0=S0,
    K=K,
    T=T,
    r=r,
    initial_sigma=0.20
)

print("Volatilité implicite :", implied_sigma_put)

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.pricing import black_scholes_call

# Paramètres
S0 = 100
T = 1
r = 0.05

# Différents strikes
strikes = np.array([80, 85, 90, 95, 100,
                    105, 110, 115, 120])

# Volatilité utilisée pour chaque strike
market_volatilities = (
    0.20 + 0.0002 * (strikes - 100)**2
)

# Génération des prix des Calls
market_prices = []

for K, sigma in zip(strikes, market_volatilities):

    price = black_scholes_call(
        S0, K, T, r, sigma
    )

    market_prices.append(price)

# Tableau récapitulatif
data = pd.DataFrame({
    "Strike": strikes,
    "Volatilite utilisee": market_volatilities,
    "Prix du Call": market_prices
})

display(data.round(6))


In [ ]:

from src.volatility import implied_volatility_call

implied_volatilities = []

for K, price in zip(strikes, market_prices):

    sigma_imp = implied_volatility_call(
        market_price=price,
        S0=S0,
        K=K,
        T=T,
        r=r,
        initial_sigma=0.20
    )

    implied_volatilities.append(sigma_imp)

# Comparaison
data["Volatilite implicite"] = implied_volatilities

display(data.round(6))


In [ ]:

plt.figure(figsize=(10, 6))

plt.plot(
    strikes,
    implied_volatilities,
    marker="o",
    label="Volatilité implicite"
)

plt.axvline(
    S0,
    linestyle="--",
    label="Prix du sous-jacent"
)

plt.xlabel("Strike K")
plt.ylabel("Volatilité implicite")

plt.title("Volatility Smile - Black-Scholes")

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig("../figures/volatility_smile.png", dpi=300)

plt.show()


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.pricing import black_scholes_call

# Paramètres
S0 = 100
T = 1
r = 0.05

# Strikes
skew_strikes = np.array([
    80, 85, 90, 95, 100, 105, 110, 115, 120
])

# Volatilités asymétriques
skew_volatilities = (
    0.20 - 0.001 * (skew_strikes - 100)
)

# Calcul des prix
skew_prices = []

for K, sigma in zip(skew_strikes, skew_volatilities):

    price = black_scholes_call(
        S0, K, T, r, sigma
    )

    skew_prices.append(price)

# Tableau
skew_data = pd.DataFrame({
    "Strike": skew_strikes,
    "Volatilite utilisee": skew_volatilities,
    "Prix du Call": skew_prices
})

display(skew_data.round(6))


In [ ]:

from src.volatility import implied_volatility_call

skew_implied_volatilities = []

for K, price in zip(skew_strikes, skew_prices):

    sigma_imp = implied_volatility_call(
        market_price=price,
        S0=S0,
        K=K,
        T=T,
        r=r,
        initial_sigma=0.20
    )

    skew_implied_volatilities.append(sigma_imp)

skew_data["Volatilite implicite"] = (
    skew_implied_volatilities
)

display(skew_data.round(6))


In [ ]:

plt.figure(figsize=(10, 6))

plt.plot(
    skew_strikes,
    skew_implied_volatilities,
    marker="o",
    label="Volatilité implicite"
)

plt.axvline(
    S0,
    linestyle="--",
    label="Prix du sous-jacent"
)

plt.xlabel("Strike K")
plt.ylabel("Volatilité implicite")

plt.title("Volatility Skew - Black-Scholes")

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig("../figures/volatility_skew.png", dpi=300)

plt.show()


In [ ]:

import yfinance as yf
import pandas as pd
import numpy as np

# Création de l'objet Apple
ticker = yf.Ticker("AAPL")

# Historique récent du cours
history = ticker.history(period="5d")

# Dernier cours de clôture disponible
S0 = float(history["Close"].dropna().iloc[-1])

# Dates d'expiration des options
expirations = ticker.options

print("ANALYSE APPLE (AAPL)")
print("--------------------")

print("Dernier cours disponible :", round(S0, 2), "$")

print("\nDates d'expiration disponibles :")

for date in expirations:
    print(date)

print("\nNombre d'expirations :", len(expirations))


In [ ]:

# Choix de l'expiration
expiration = "2026-12-18"

# Vérification de la disponibilité
if expiration not in expirations:
    raise ValueError("Date d'expiration indisponible.")

# Récupération de la chaîne d'options
option_chain = ticker.option_chain(expiration)

# Séparation des Calls et des Puts
calls = option_chain.calls.copy()
puts = option_chain.puts.copy()

print("Nombre de Calls :", len(calls))
print("Nombre de Puts :", len(puts))

# Affichage des premières observations
display(calls.head())
display(puts.head())


In [ ]:

# Calcul du prix moyen Bid-Ask
calls["mid_price"] = (calls["bid"] + calls["ask"]) / 2
puts["mid_price"] = (puts["bid"] + puts["ask"]) / 2

# Conservation des cotations exploitables
calls_clean = calls[
    (calls["bid"] > 0) &
    (calls["ask"] >= calls["bid"]) &
    (calls["ask"] > 0)
].copy()

puts_clean = puts[
    (puts["bid"] > 0) &
    (puts["ask"] >= puts["bid"]) &
    (puts["ask"] > 0)
].copy()

# Sélection des strikes proches du cours Apple
lower_bound = 0.80 * S0
upper_bound = 1.20 * S0

calls_clean = calls_clean[
    calls_clean["strike"].between(lower_bound, upper_bound)
]

puts_clean = puts_clean[
    puts_clean["strike"].between(lower_bound, upper_bound)
]

print("CALLS exploitables :")
display(calls_clean[
    ["strike", "bid", "ask", "mid_price",
     "impliedVolatility", "volume", "openInterest"]
].head(15))

print("PUTS exploitables :")
display(puts_clean[
    ["strike", "bid", "ask", "mid_price",
     "impliedVolatility", "volume", "openInterest"]
].head(15))


In [ ]:

# Calcul de la maturité

valuation_date = history.index[-1].date()
expiration_date = pd.to_datetime(expiration).date()

days_to_expiration = (
    expiration_date - valuation_date
).days

T = days_to_expiration / 365

print("Date de référence :", valuation_date)
print("Date d'expiration :", expiration_date)
print("Nombre de jours :", days_to_expiration)
print("Maturité T :", T)


In [ ]:

# Définition de la zone proche de la monnaie

lower_bound = 0.95 * S0
upper_bound = 1.05 * S0

# Sélection des Calls
calls_atm = calls_clean[
    calls_clean["strike"].between(
        lower_bound, upper_bound
    )
].copy()

# Sélection des Puts
puts_atm = puts_clean[
    puts_clean["strike"].between(
        lower_bound, upper_bound
    )
].copy()

# Tri par strike
calls_atm = calls_atm.sort_values("strike")
puts_atm = puts_atm.sort_values("strike")

print("CALLS proches de la monnaie")
display(calls_atm[
    ["strike", "bid", "ask", "mid_price",
     "impliedVolatility", "volume", "openInterest"]
])

print("PUTS proches de la monnaie")
display(puts_atm[
    ["strike", "bid", "ask", "mid_price",
     "impliedVolatility", "volume", "openInterest"]
])


In [ ]:

from src.volatility import (
    implied_volatility_call,
    implied_volatility_put
)

# Taux sans risque retenu comme hypothèse
r = 0.05

# Fonction de calcul des volatilités implicites
def calculate_market_iv(data, option_type):

    data = data.copy()
    implied_vols = []

    for _, row in data.iterrows():

        K = float(row["strike"])
        market_price = float(row["mid_price"])

        try:

            if option_type == "call":

                sigma = implied_volatility_call(
                    market_price,
                    S0,
                    K,
                    T,
                    r
                )

            elif option_type == "put":

                sigma = implied_volatility_put(
                    market_price,
                    S0,
                    K,
                    T,
                    r
                )

            else:
                raise ValueError("Type inconnu")

        except (ValueError, RuntimeError, OverflowError):
            sigma = np.nan

        implied_vols.append(sigma)

    data["iv_model"] = implied_vols

    # Comparaison avec Yahoo Finance
    data["iv_yahoo"] = data["impliedVolatility"]

    data["ecart"] = (
        data["iv_model"] - data["iv_yahoo"]
    )

    return data


# Calcul pour les Calls et les Puts
calls_results = calculate_market_iv(
    calls_atm, "call"
)

puts_results = calculate_market_iv(
    puts_atm, "put"
)

print("RESULTATS CALLS")
display(calls_results[
    ["strike", "mid_price", "iv_yahoo",
     "iv_model", "ecart"]
].round(6))

print("RESULTATS PUTS")
display(puts_results[
    ["strike", "mid_price", "iv_yahoo",
     "iv_model", "ecart"]
].round(6))


In [ ]:

plt.figure(figsize=(10, 6))

plt.plot(
    calls_results["strike"],
    calls_results["iv_yahoo"] * 100,
    marker="o",
    label="Yahoo Finance"
)

plt.plot(
    calls_results["strike"],
    calls_results["iv_model"] * 100,
    marker="s",
    linestyle="--",
    label="Notre modèle Black-Scholes"
)

plt.axvline(S0, linestyle=":", label="Prix Apple")

plt.xlabel("Strike ($)")
plt.ylabel("Volatilité implicite (%)")
plt.title("Apple - Comparaison des volatilités implicites des Calls")

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig("../figures/aapl_calls_comparison.png", dpi=300)
plt.show()


In [ ]:

plt.figure(figsize=(10, 6))

plt.plot(
    puts_results["strike"],
    puts_results["iv_yahoo"] * 100,
    marker="o",
    label="Yahoo Finance"
)

plt.plot(
    puts_results["strike"],
    puts_results["iv_model"] * 100,
    marker="s",
    linestyle="--",
    label="Notre modèle Black-Scholes"
)

plt.axvline(S0, linestyle=":", label="Prix Apple")

plt.xlabel("Strike ($)")
plt.ylabel("Volatilité implicite (%)")
plt.title("Apple - Comparaison des volatilités implicites des Puts")

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig("../figures/aapl_puts_comparison.png", dpi=300)
plt.show()


In [ ]:

# Calcul de l'erreur absolue moyenne

mae_calls = np.mean(
    np.abs(
        calls_results["iv_model"] -
        calls_results["iv_yahoo"]
    )
)

mae_puts = np.mean(
    np.abs(
        puts_results["iv_model"] -
        puts_results["iv_yahoo"]
    )
)

print("COMPARAISON DES MODELES")
print("-----------------------")

print("MAE Calls :", round(mae_calls, 6))
print("MAE Puts  :", round(mae_puts, 6))


In [ ]:


from src.hedging import simulate_gbm_path
import matplotlib.pyplot as plt

S0 = 100
r = 0.05
sigma = 0.20
T = 1

times, prices = simulate_gbm_path(
    S0=S0,
    r=r,
    sigma=sigma,
    T=T,
    n_steps=252,
    seed=42
)

print("Prix initial :", prices[0])
print("Prix final :", round(prices[-1], 4))
print("Prix minimum :", round(prices.min(), 4))
print("Prix maximum :", round(prices.max(), 4))

plt.figure(figsize=(10, 5))

plt.plot(times, prices)

plt.axhline(
    S0,
    linestyle="--",
    label="Prix initial"
)

plt.xlabel("Temps (années)")
plt.ylabel("Prix de l'action (€)")
plt.title("Simulation du prix de l'action - GBM")

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig("../figures/gbm_simulation.png", dpi=300)
plt.show()


In [ ]:
# Rechargement du module pour utiliser la version actuelle du dépôt
import importlib
import src.hedging as hedging
importlib.reload(hedging)

S = 100
K = 100
tau = 1
r = 0.05
sigma = 0.20

delta = hedging.calculate_call_delta(S, K, tau, r, sigma)

print('Delta du Call :', delta)
print('Nombre d actions nécessaires pour couvrir un Call vendu :', delta)

## 7. Étude de la couverture dynamique Delta Hedging

Nous considérons une position courte sur un Call européen, couverte par un portefeuille composé du sous-jacent et d'un compte de trésorerie. Les trajectoires du sous-jacent suivent un mouvement brownien géométrique sous la mesure risque-neutre. Les paramètres et la graine aléatoire sont identiques entre les stratégies afin de comparer les fréquences sur une même trajectoire.

La stratégie est rééquilibrée quotidiennement, hebdomadairement (tous les 5 pas) ou mensuellement (tous les 21 pas). Les coûts de transaction sont exprimés en points de base (bps) et appliqués au notionnel échangé, y compris lors de la mise en place de la couverture initiale.

La valeur du portefeuille est calculée comme :

$$V_t = B_t + \Delta_t^{hedge} S_t - C_t,$$

où $B_t$ est la trésorerie, $\Delta_t^{hedge}$ le nombre d'actions détenues et $C_t$ la valeur du Call vendu (sa valeur intrinsèque à l'échéance). À l'échéance, la position en actions est valorisée au cours final ; elle n'est pas forcée à zéro dans le moteur de simulation.

In [ ]:
import importlib
import src.hedging as hedging
importlib.reload(hedging)
import pandas as pd
import matplotlib.pyplot as plt

# Paramètres communs à toutes les stratégies
S0 = 100.0
K = 100.0
T = 0.5                 # maturité de six mois
r = 0.05
sigma = 0.20            # volatilité constante du modèle
n_steps = 126           # environ 252 séances par an sur six mois
seed = 42
frequencies = ["daily", "weekly", "monthly"]
cost_scenarios_bps = [0.0, 5.0, 10.0]

simulations = {}
summary = []

for cost_bps in cost_scenarios_bps:
    for frequency in frequencies:
        result = hedging.delta_hedging_simulation(
            S0=S0, K=K, T=T, r=r, sigma=sigma,
            n_steps=n_steps, seed=seed,
            rebalance_frequency=frequency,
            transaction_cost_bps=cost_bps,
        )
        simulations[(frequency, cost_bps)] = result
        summary.append({
            "Fréquence": frequency,
            "Coût (bps)": cost_bps,
            "P&L final": result["portfolio_value"].iloc[-1],
            "Coûts cumulés": result["cumulative_transaction_cost"].iloc[-1],
            "Nombre de rééquilibrages": int(result["rebalanced"].sum()),
            "Erreur quadratique moyenne": (result["portfolio_value"] ** 2).mean() ** 0.5,
        })

summary_df = pd.DataFrame(summary)
display(summary_df.round(4))

### Comparaison des trajectoires sans coûts de transaction

On commence par isoler l'effet de la fréquence de rééquilibrage : les trois stratégies utilisent la même trajectoire du sous-jacent et ne supportent aucun coût de transaction.

In [ ]:
plt.figure(figsize=(11, 6))
for frequency in frequencies:
    result = simulations[(frequency, 0.0)]
    plt.plot(result["time"], result["portfolio_value"], label=frequency.capitalize())

plt.axhline(0, linestyle="--", linewidth=1, label="P&L nul")
plt.xlabel("Temps (années)")
plt.ylabel("Valeur du portefeuille de couverture")
plt.title("P&L de couverture selon la fréquence de rééquilibrage — sans coûts")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("../figures/delta_hedging_frequency.png", dpi=300)
plt.show()

### Impact des coûts de transaction

Les coûts augmentent avec le notionnel échangé et la fréquence des ajustements. On compare ici des coûts de 0, 5 et 10 bps. Les résultats dépendent de la trajectoire simulée, des paramètres du modèle et de la convention de coûts retenue ; ils ne constituent pas une estimation universelle des coûts de marché.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for frequency in frequencies:
    rows = summary_df[summary_df["Fréquence"] == frequency].sort_values("Coût (bps)")
    axes[0].plot(rows["Coût (bps)"], rows["P&L final"], marker="o", label=frequency.capitalize())
    axes[1].plot(rows["Coût (bps)"], rows["Coûts cumulés"], marker="o", label=frequency.capitalize())

axes[0].set_title("P&L final selon les coûts")
axes[0].set_xlabel("Coût de transaction (bps)")
axes[0].set_ylabel("P&L final")
axes[1].set_title("Coûts de transaction cumulés")
axes[1].set_xlabel("Coût de transaction (bps)")
axes[1].set_ylabel("Coûts cumulés")
for ax in axes:
    ax.grid(alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.savefig("../figures/delta_hedging_transaction_costs.png", dpi=300)
plt.show()

### Lecture financière et limites

- À volatilité réalisée égale à la volatilité utilisée pour valoriser le Call, la couverture discrète conserve une erreur résiduelle : elle ne reproduit pas la couverture continue théorique.
- Une fréquence de rééquilibrage plus élevée réduit généralement le risque de couverture discret, mais augmente le nombre d'opérations et donc les coûts.
- Les coûts de transaction peuvent modifier le classement des stratégies ; il faut comparer simultanément le P&L résiduel et les coûts cumulés.
- Le modèle suppose ici un taux et une volatilité constants, aucun dividende, aucune contrainte de liquidité et une trajectoire simulée sous la mesure risque-neutre.

Les résultats numériques doivent être interprétés à partir du tableau et des graphiques générés lors de l'exécution du notebook.